
# IAMC Tools Scraper → Consolidated Excel (v3: Dynamic columns from `project_details`)

This notebook scrapes IAM Consortium **Tool Resources** pages and builds a consolidated **Excel (.xlsx)** file.

**What's new in v3**
- **Description** is taken **only** from `<p>` elements (prefers `.project-details p`, else falls back to `#page-content p`).
- The Excel/CSV table now **includes every unique field** present in `project_details` across all tools.  
  That is, we compute the union of keys from all `project_details` dicts and make a column for each one.

Created using Chat-GPT and further adapted to produce the required results 

In [9]:

import json
import time
from typing import List, Dict, Any, Set
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup
import pandas as pd
from pathlib import Path
from pandas import ExcelWriter


def fetch_html(url: str, timeout: int = 25) -> str:
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        )
    }
    resp = requests.get(url, headers=headers, timeout=timeout)
    resp.raise_for_status()
    return resp.text


def clean_text(text: str) -> str:
    return " ".join(text.split())


def extract_links(container: BeautifulSoup, base_url: str):
    links = []
    for a in container.find_all("a", href=True):
        href = a["href"].strip()
        abs_url = urljoin(base_url, href)
        label = a.get_text(" ", strip=True)
        links.append({"text": label, "href": abs_url})
    return links


def extract_description(container: BeautifulSoup) -> str:
    """Return ONLY text from <p> elements.
    Priority: p's within div.project-details; fallback: p's within the container (#page-content).
    """
    pdv = container.find("div", class_="project-details")
    paragraphs = []
    if pdv:
        for p in pdv.find_all("p"):
            txt = clean_text(p.get_text(" ", strip=True))
            if txt:
                paragraphs.append(txt)
    if not paragraphs:
        for p in container.find_all("p"):
            txt = clean_text(p.get_text(" ", strip=True))
            if txt:
                paragraphs.append(txt)
    return " ".join(paragraphs)


def extract_project_details(container: BeautifulSoup) -> Dict[str, Any]:
    details: Dict[str, Any] = {}
    pdv = container.find("div", class_="project-details")
    if not pdv:
        return details

    title_block = pdv.find(class_="page-title-txt")
    if title_block:
        h = title_block.find(["h1", "h2", "h3", "h4", "h5", "h6"])
        if h:
            details["Title"] = clean_text(h.get_text(" ", strip=True))

    ul = pdv.find("ul", class_="project-details-list")
    if not ul:
        return details

    for li in ul.find_all("li"):
        label_span = li.find("span", class_="detail-label")
        value_span = li.find("span", class_="detail-value")
        if not label_span or not value_span:
            key = clean_text(li.get_text(" ", strip=True))
            if key:
                details[key] = True
            continue

        key = clean_text(label_span.get_text(" ", strip=True)).rstrip(":")
        for br in value_span.find_all("br"):
            br.replace_with("\n")
        raw_value = value_span.get_text("\n", strip=True)

        if "\n" in raw_value:
            parts = [clean_text(p) for p in raw_value.split("\n") if clean_text(p)]
            value = parts
        else:
            value = clean_text(raw_value)

        details[key] = value

    return details


def extract_page_content(url: str) -> Dict[str, Any]:
    html = fetch_html(url)
    soup = BeautifulSoup(html, "html.parser")

    container = soup.find(id="page-content")
    if not container:
        raise RuntimeError("Could not find <div id='page-content'> on the page.")

    full_text = clean_text(container.get_text(" ", strip=True))
    description = extract_description(container)
    links = extract_links(container, url)
    project_details = extract_project_details(container)

    page_title = None
    possible_title = container.find(["h1", "h2", "h3"])
    if possible_title:
        page_title = clean_text(possible_title.get_text(" ", strip=True))

    return {
        "url": url,
        "title": page_title,
        "description": description,  # p-only description
        "text": full_text,          # reference
        "links": links,
        "project_details": project_details,
    }


def normalize(value):
    if isinstance(value, list):
        return "; ".join(v for v in value if v)
    return value if value is not True else ""


BASE_COLUMNS = [
    "URL",
    "Title",
    "Description",
    "First link in content",
]


def collect_all_project_detail_keys(items: List[Dict[str, Any]]) -> List[str]:
    seen: Set[str] = set()
    for it in items:
        pdict = it.get("project_details") or {}
        for k in pdict.keys():
            seen.add(k)
    # We'll prefer a friendly order: common fields first, then the rest sorted.
    preferred = [
        "Geographical scope",
        "Institution(s)",
        "Users",
        "Link",
        "Contact",
        "Contact e-mail",
    ]
    ordered = [k for k in preferred if k in seen]
    remaining = sorted(k for k in seen if k not in ordered)
    return ordered + remaining


def to_flat_row_dynamic(data: Dict[str, Any], all_keys: List[str]) -> Dict[str, Any]:
    pdict = data.get("project_details") or {}
    row = {
        "URL": data.get("url", ""),
        "Title": pdict.get("Title", data.get("title", "")),
        "Description": normalize(data.get("description", "")),  # p-only
        "First link in content": (data.get("links") or [{}])[0].get("href", ""),
    }
    # Add dynamic project_detail keys (skip if would overwrite a base column)
    for k in all_keys:
        if k in row:  # avoid clobbering base columns (e.g., TOOL)
            continue
        row[k] = normalize(pdict.get(k, ""))
    return row



## 1) Provide your URL list
Use either the inline list or place them in `tool_urls.csv` (one URL per line).


In [10]:

# Inline list (edit as needed)
URLS = [
    "https://www.iamconsortium.org/resources/tool-resources/atlas-of-climate-policy-barriers-2/",
    # "https://www.iamconsortium.org/resources/tool-resources/carbon-monitor/",
]
len(URLS)


1

In [11]:

# Optionally load from tool_urls.csv (one URL per line)
csv_path = Path("original/iamc_tools_clean.txt")
if csv_path.exists():
    df_urls = pd.read_csv(csv_path, header=None, names=["url"])
    URLS = [u for u in df_urls["url"].astype(str).tolist() if u.strip()]
    print(f"Loaded {len(URLS)} URLs from tool_urls.csv")
else:
    print("No tool_urls.csv found; using URLs from the previous cell.")


Loaded 46 URLs from tool_urls.csv



## 2) Scrape pages
This iterates over all URLs and collects JSON entries.


In [12]:

all_json: List[Dict[str, Any]] = []
failures = []

for i, url in enumerate(URLS, start=1):
    try:
        data = extract_page_content(url)
        all_json.append(data)
        print(f"[{i}/{len(URLS)}] OK: {url}")
    except Exception as e:
        failures.append({"url": url, "error": str(e)})
        print(f"[{i}/{len(URLS)}] ERROR: {url} -> {e}")
    time.sleep(0.8)  # politeness delay

print("\nDone. Success:", len(all_json), "Failures:", len(failures))


[1/46] OK: https://www.iamconsortium.org/resources/tool-resources/atlas-of-climate-policy-barriers-2/
[2/46] OK: https://www.iamconsortium.org/resources/tool-resources/carbon-monitor/
[3/46] OK: https://www.iamconsortium.org/resources/tool-resources/ceds-community-earth-atmospheric-data-system/
[4/46] OK: https://www.iamconsortium.org/news-from-the-community/news-f-the-community/circular-carbon-economy-index-2022-kapsarc/
[5/46] OK: https://www.iamconsortium.org/resources/tool-resources/climate-action-tracker/
[6/46] OK: https://www.iamconsortium.org/resources/tool-resources/climate-policy-database/
[7/46] OK: https://www.iamconsortium.org/news-from-the-community/news-f-the-community/climate-risk-dashboard/
[8/46] OK: https://www.iamconsortium.org/resources/tool-resources/climate-watch-net-zero-tracker/
[9/46] OK: https://www.iamconsortium.org/news-from-the-community/news-f-the-community/coacch-scenario-explorer/
[10/46] OK: https://www.iamconsortium.org/news-from-the-community/news-f-


## 3) Build dynamic table and save JSONL, CSV, and Excel
- The union of keys from `project_details` becomes columns.
- **JSONL**: `iamc_tools_v3.jsonl`
- **CSV**: `iamc_tools_v3.csv`
- **Excel**: `iamc_tools_v3.xlsx` (sheets: `tools`, `raw_json`)


In [13]:

from pathlib import Path
import json
import pandas as pd

out_dir = Path(".")
jsonl_path = out_dir / "iamc_tools_v3.jsonl"
csv_path = out_dir / "iamc_tools_v3.csv"
xlsx_path = out_dir / "iamc_tools_v3.xlsx"

# # JSONL
# with jsonl_path.open("w", encoding="utf-8") as f:
#     for obj in all_json:
#         f.write(json.dumps(obj, ensure_ascii=False) + "\n")

# Dynamic columns from all project_details
all_keys = collect_all_project_detail_keys(all_json)

# Build flat rows with dynamic columns
rows = [to_flat_row_dynamic(obj, all_keys) for obj in all_json]
df = pd.DataFrame(rows, columns=BASE_COLUMNS + all_keys)  # keeps ordering

# Save CSV & Excel
# df.to_csv(csv_path, index=False, encoding="utf-8")

with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    df.to_excel(writer, index=False, sheet_name="tools")
    raw_df = pd.DataFrame([{
        "url": j["url"],
        "title": j.get("title", ""),
        "description": j.get("description", ""),
        "project_details": json.dumps(j.get("project_details", {}), ensure_ascii=False),
        "links": json.dumps(j.get("links", []), ensure_ascii=False),
    } for j in all_json])
    raw_df.to_excel(writer, index=False, sheet_name="raw_json")

jsonl_path, csv_path, xlsx_path, all_keys


(PosixPath('iamc_tools_v3.jsonl'),
 PosixPath('iamc_tools_v3.csv'),
 PosixPath('iamc_tools_v3.xlsx'),
 ['Geographical scope',
  'Institution(s)',
  'Users',
  'Link',
  'Contact',
  'Contact e-mail',
  'Initial Release',
  'Model type',
  'Time horizon',
  'Title'])


## 4) Inspect failures (optional)


In [14]:

import pandas as pd

if failures:
    display(pd.DataFrame(failures))
else:
    print("No failures.")


,url,error
0,https://www.iamconsortium.org/resources/tool-r...,404 Client Error: Not Found for url: https://w...


In [15]:
failures

[{'url': 'https://www.iamconsortium.org/resources/tool-resources/12146/',
  'error': '404 Client Error: Not Found for url: https://www.iamconsortium.org/resources/tool-resources/12146/'}]